<a href="https://colab.research.google.com/github/TNGBBK/ChatRTX/blob/release%2F0.4.0/phi2Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython import get_ipython
from IPython.display import display

In [2]:
!pip install transformers datasets accelerate bitsandbytes peft trl wandb optimum pyyaml graphviz pydot tflite-runtime
!apt-get install libedgetpu1-max


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libedgetpu1-max


In [3]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [4]:
!pip install tensorflow-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.2/230.2 MB 4.6 MB/s eta 0:00:00


In [13]:
# Imports ที่จำเป็น
import torch
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import bitsandbytes as bnb
import wandb
import yaml
import os
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import train_test_split

In [14]:
# %%
# ตรวจสอบว่า Colab ใช้ TPU หรือไม่
if 'TPU_NAME' in os.environ:
    print("TPU detected!")
    import torch_xla.distributed.xla_backend
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
else:
    print("No TPU detected, using CPU/GPU instead.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

No TPU detected, using CPU/GPU instead.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
%%writefile config.yaml
model:
  model_id: "microsoft/phi-2"
  tokenizer_id: "microsoft/phi-2"
  quantization_method: "4bit"
  quantization_type: "nf4"
  lora: True

dataset:
  instruction_dataset: "databricks/databricks-dolly-15k"
  mindmap_dataset: "databricks/databricks-dolly-15k"
  data_source: "huggingface"
  preprocessing:
    max_length: 512

training:
  output_dir: "phi2-mindmap-chatbot"
  experiment_name: "phi2-instruction-mindmap"
  seed: 42
  batch_size: 8  # ปรับ Batch Size ให้เหมาะสมกับ TPU
  gradient_accumulation_steps: 8  # ปรับ Gradient Accumulation Steps ให้เหมาะสมกับ TPU
  learning_rate: 2.0e-4
  weight_decay: 0.01
  num_epochs: 3
  logging_steps: 10
  eval_steps: 50
  save_steps: 50
  evaluation_strategy: "steps"
  save_strategy: "steps"
  fp16: True
  lora_r: 8
  lora_alpha: 16
  lora_dropout: 0.05
  lora_target_modules: ["Wqkv", "out_proj"]

quantization:
  quantization_config: {}

pruning:
  pruning_config: {}

evaluation:
  metrics: ["perplexity", "bleu", "rouge"]

Overwriting config.yaml


In [16]:
# ฟังก์ชัน load config
def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

# ฟังก์ชัน load dataset
def load_dataset_from_huggingface(dataset_name):
    return load_dataset(dataset_name)

# ฟังก์ชัน tokenize
def map_preprocessing(dataset, tokenizer_name, max_length):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def tokenize_function(examples):
        return tokenizer(examples["instruction"], truncation=True, max_length=max_length, padding="max_length", return_tensors="pt")

    return dataset.map(tokenize_function, batched=True)

In [17]:
# ฟังก์ชัน Data Augmentation
def augment_data(dataset, alpha=0.05):
    def augment_function(examples):
        augmented_instructions = []
        for instruction in examples["instruction"]:
            words = instruction.split()
            num_words_to_replace = max(1, int(alpha * len(words)))  # Ensure at least one word is replaced
            indices_to_replace = np.random.choice(len(words), num_words_to_replace, replace=False)
            for i in indices_to_replace:
                words[i] = np.random.choice(words)  # Replace with a random word from the same instruction
            augmented_instructions.append(" ".join(words))
        return {"instruction": augmented_instructions}

    return dataset.map(augment_function, batched=True)

# ฟังก์ชัน load model และ tokenizer
def load_model_and_tokenizer(model_id, config, lora_adapter_path=None):
    quantization_method = config['model']['quantization_method']
    bnb_config = None
    if quantization_method == "4bit":
        bnb_config = bnb.BitsAndBytesConfig(  # แก้ไข: เพิ่ม bnb. ก่อน BitsAndBytesConfig
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type=config['model']['quantization_type'],
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(config["model"]["tokenizer_id"])
    tokenizer.pad_token = tokenizer.eos_token

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    if lora_adapter_path:
        model = PeftModel.from_pretrained(model, lora_adapter_path)

    return model, tokenizer

In [18]:
# ฟังก์ชัน load model และ tokenizer
def load_model_and_tokenizer(model_id, config, lora_adapter_path=None):
    quantization_method = config['model']['quantization_method']
    bnb_config = None
    if quantization_method == "4bit":
        bnb_config = bnb.BitsAndBytesConfig(  # แก้ไข: เพิ่ม bnb. ก่อน BitsAndBytesConfig
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type=config['model']['quantization_type'],
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(config["model"]["tokenizer_id"])
    tokenizer.pad_token = tokenizer.eos_token

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    if lora_adapter_path:
        model = PeftModel.from_pretrained(model, lora_adapter_path)

    return model, tokenizer

# ฟังก์ชัน training
def train_model(model, tokenizer, train_dataset, eval_dataset, config):
    lora_config = LoraConfig(
        r=config['training']['lora_r'],
        lora_alpha=config['training']['lora_alpha'],
        target_modules=config['training']['lora_target_modules'],
        lora_dropout=config['training']['lora_dropout'],
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    training_args = TrainingArguments(
        output_dir=config['training']['output_dir'],
        per_device_train_batch_size=config['training']['batch_size'],
        gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
        learning_rate=config['training']['learning_rate'],
        weight_decay=config['training']['weight_decay'],
        fp16=config['training']['fp16'],
        logging_steps=config['training']['logging_steps'],
        evaluation_strategy=config['training']['evaluation_strategy'],
        eval_steps=config['training']['eval_steps'],
        save_strategy=config['training']['save_strategy'],
        save_steps=config['training']['save_steps'],
        num_train_epochs=config['training']['num_epochs'],
        seed=config['training']['seed'],
        report_to="wandb",
        dataloader_num_workers=0,  # Important for TPU
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    trainer.train()
    return model
# %%
# Load the configuration
config = load_config('config.yaml')

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(config["model"]["tokenizer_id"])

# กำหนด padding token เป็น eos_token
tokenizer.pad_token = tokenizer.eos_token

# Define input_text here
input_text = "This is an example input." # Replace with your desired input

# Now you can use the tokenizer
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)